In [1]:
import pandas as pd
import os

# get all csv files in current directory
csv_files = [f for f in os.listdir() if f.endswith('.csv')]

csv_files


['Bot_IoT_DDoS.csv', 'CombineFileAnCa171813.csv']

In [2]:
dfs = {}

for file in csv_files:
    dfs[file] = pd.read_csv(file)


In [3]:
for file, df in dfs.items():
    print(f"\n📄 {file}")
    print(list(df.columns))



📄 Bot_IoT_DDoS.csv
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 

In [4]:
for file, df in dfs.items():
    print(f"\n📄 {file}")
    print(df.dtypes)



📄 Bot_IoT_DDoS.csv
Flow ID       object
Src IP        object
Src Port       int64
Dst IP        object
Dst Port       int64
              ...   
Idle Mean    float64
Idle Std     float64
Idle Max     float64
Idle Min     float64
Label         object
Length: 84, dtype: object

📄 CombineFileAnCa171813.csv
Flow ID       object
Src IP        object
Src Port     float64
Dst IP        object
Dst Port     float64
              ...   
Idle Mean    float64
Idle Std     float64
Idle Max     float64
Idle Min     float64
Label          int64
Length: 84, dtype: object


In [5]:
schema_comparison = pd.DataFrame()

for file, df in dfs.items():
    schema_comparison[file] = df.dtypes

schema_comparison


,Bot_IoT_DDoS.csv,CombineFileAnCa171813.csv
Flow ID,object,object
Src IP,object,object
Src Port,int64,float64
Dst IP,object,object
Dst Port,int64,float64
...,...,...
Idle Mean,float64,float64
Idle Std,float64,float64
Idle Max,float64,float64
Idle Min,float64,float64


In [6]:
incompatible_columns = schema_comparison.nunique(axis=1)
incompatible_columns[incompatible_columns > 1]


Src Port             2
Dst Port             2
Protocol             2
Flow Duration        2
Tot Fwd Pkts         2
Tot Bwd Pkts         2
Fwd PSH Flags        2
Bwd PSH Flags        2
Fwd URG Flags        2
Bwd URG Flags        2
Fwd Header Len       2
Bwd Header Len       2
FIN Flag Cnt         2
SYN Flag Cnt         2
RST Flag Cnt         2
PSH Flag Cnt         2
ACK Flag Cnt         2
URG Flag Cnt         2
CWE Flag Count       2
ECE Flag Cnt         2
Fwd Byts/b Avg       2
Fwd Pkts/b Avg       2
Fwd Blk Rate Avg     2
Bwd Byts/b Avg       2
Bwd Pkts/b Avg       2
Bwd Blk Rate Avg     2
Subflow Fwd Pkts     2
Subflow Fwd Byts     2
Subflow Bwd Pkts     2
Subflow Bwd Byts     2
Init Fwd Win Byts    2
Init Bwd Win Byts    2
Fwd Act Data Pkts    2
Fwd Seg Size Min     2
Label                2
dtype: int64

In [7]:
all_columns = set().union(*[df.columns for df in dfs.values()])

for file, df in dfs.items():
    missing = all_columns - set(df.columns)
    if missing:
        print(f"\n⚠️ {file} is missing columns: {missing}")



⚠️ Bot_IoT_DDoS.csv is missing columns: {'Bwd Packets/s', 'Bwd IAT Total', 'Fwd Packets/s'}

⚠️ CombineFileAnCa171813.csv is missing columns: {'Bwd Pkts/s', 'Fwd Pkts/s', 'Bwd IAT Tot'}


In [8]:
COLUMN_RENAME_MAP_REVERSED = {
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
}


In [9]:
for file, df in dfs.items():
    df.rename(columns=COLUMN_RENAME_MAP_REVERSED, inplace=True)


In [10]:
dfs['Bot_IoT_DDoS.csv']['Label'].value_counts()

Label
No Label    38049077
Name: count, dtype: int64

In [12]:
dfs['Bot_IoT_DDoS.csv']['Label'] = 1

In [13]:
dfs['Bot_IoT_DDoS.csv']['Label'].value_counts()

Label
1    38049077
Name: count, dtype: int64

In [14]:
all_columns = set().union(*[df.columns for df in dfs.values()])
len(all_columns)


84

In [15]:
for file, df in dfs.items():
    missing = all_columns - set(df.columns)
    extra = set(df.columns) - all_columns

    if not missing and not extra:
        print(f"✅ {file}: column names fully compatible")
    else:
        print(f"\n⚠️ {file}")
        if missing:
            print(f"  Missing columns: {missing}")
        if extra:
            print(f"  Extra columns: {extra}")


✅ Bot_IoT_DDoS.csv: column names fully compatible
✅ CombineFileAnCa171813.csv: column names fully compatible


In [16]:
base_file = list(dfs.keys())[0]
base_columns = list(dfs[base_file].columns)

for file, df in dfs.items():
    if list(df.columns) == base_columns:
        print(f"✅ {file}: exact column match with {base_file}")
    else:
        print(f"❌ {file}: column order or names differ from {base_file}")


✅ Bot_IoT_DDoS.csv: exact column match with Bot_IoT_DDoS.csv
✅ CombineFileAnCa171813.csv: exact column match with Bot_IoT_DDoS.csv


In [17]:
dtype_table = pd.DataFrame(
    {file: df.dtypes for file, df in dfs.items()}
)

dtype_table


,Bot_IoT_DDoS.csv,CombineFileAnCa171813.csv
Flow ID,object,object
Src IP,object,object
Src Port,int64,float64
Dst IP,object,object
Dst Port,int64,float64
...,...,...
Idle Mean,float64,float64
Idle Std,float64,float64
Idle Max,float64,float64
Idle Min,float64,float64


In [18]:
dtype_mismatch = dtype_table.nunique(axis=1)
dtype_mismatch[dtype_mismatch > 1]


Src Port             2
Dst Port             2
Protocol             2
Flow Duration        2
Tot Fwd Pkts         2
Tot Bwd Pkts         2
Fwd PSH Flags        2
Bwd PSH Flags        2
Fwd URG Flags        2
Bwd URG Flags        2
Fwd Header Len       2
Bwd Header Len       2
FIN Flag Cnt         2
SYN Flag Cnt         2
RST Flag Cnt         2
PSH Flag Cnt         2
ACK Flag Cnt         2
URG Flag Cnt         2
CWE Flag Count       2
ECE Flag Cnt         2
Fwd Byts/b Avg       2
Fwd Pkts/b Avg       2
Fwd Blk Rate Avg     2
Bwd Byts/b Avg       2
Bwd Pkts/b Avg       2
Bwd Blk Rate Avg     2
Subflow Fwd Pkts     2
Subflow Fwd Byts     2
Subflow Bwd Pkts     2
Subflow Bwd Byts     2
Init Fwd Win Byts    2
Init Bwd Win Byts    2
Fwd Act Data Pkts    2
Fwd Seg Size Min     2
dtype: int64

In [19]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

dtype_table

,Bot_IoT_DDoS.csv,CombineFileAnCa171813.csv
Flow ID,object,object
Src IP,object,object
Src Port,int64,float64
Dst IP,object,object
Dst Port,int64,float64
Protocol,int64,float64
Timestamp,object,object
Flow Duration,int64,float64
Tot Fwd Pkts,int64,float64
Tot Bwd Pkts,int64,float64


In [20]:
final_df = pd.concat(dfs.values(), ignore_index=True)


In [21]:
final_df.shape


(53077296, 84)

In [22]:

final_df.dtypes


Flow ID               object
Src IP                object
Src Port             float64
Dst IP                object
Dst Port             float64
Protocol             float64
Timestamp             object
Flow Duration        float64
Tot Fwd Pkts         float64
Tot Bwd Pkts         float64
TotLen Fwd Pkts      float64
TotLen Bwd Pkts      float64
Fwd Pkt Len Max      float64
Fwd Pkt Len Min      float64
Fwd Pkt Len Mean     float64
Fwd Pkt Len Std      float64
Bwd Pkt Len Max      float64
Bwd Pkt Len Min      float64
Bwd Pkt Len Mean     float64
Bwd Pkt Len Std      float64
Flow Byts/s          float64
Flow Pkts/s          float64
Flow IAT Mean        float64
Flow IAT Std         float64
Flow IAT Max         float64
Flow IAT Min         float64
Fwd IAT Tot          float64
Fwd IAT Mean         float64
Fwd IAT Std          float64
Fwd IAT Max          float64
Fwd IAT Min          float64
Bwd IAT Total        float64
Bwd IAT Mean         float64
Bwd IAT Std          float64
Bwd IAT Max   

In [23]:
final_df.to_csv("leave_CIC_DDoS_2019.csv", index=False)


In [24]:
final_df['Label'].value_counts()

Label
1    43431642
0     9645654
Name: count, dtype: int64

In [25]:
final_df.duplicated().sum()

np.int64(205)

In [26]:
import numpy as np

# Numeric columns only
num_df = final_df.select_dtypes(include=[np.number])

# 1️⃣ NaN
nan_rows = final_df.isna().any(axis=1).sum()
total_nans = final_df.isna().sum().sum()

# 2️⃣ Infinity
inf_rows = np.isinf(num_df).any(axis=1).sum()
total_infs = np.isinf(num_df).sum().sum()

# 3️⃣ Empty strings
# Only object (string) columns
str_df = final_df.select_dtypes(include=[object])
empty_rows = (str_df == '').any(axis=1).sum()
total_empty = (str_df == '').sum().sum()

# ✅ Print results
print(f"NaN: {nan_rows} rows, {total_nans} total values")
print(f"Infinity: {inf_rows} rows, {total_infs} total values")
print(f"Empty strings: {empty_rows} rows, {total_empty} total values")


NaN: 40318 rows, 43855 total values
Infinity: 59467 rows, 82153 total values
Empty strings: 0 rows, 0 total values


In [27]:
(final_df[['Tot Fwd Pkts', 'Tot Bwd Pkts', 'Flow Duration']] < 0).sum()


Tot Fwd Pkts       0
Tot Bwd Pkts       0
Flow Duration    115
dtype: int64

In [28]:
final_df['Flow ID'].nunique(), final_df.shape[0]


(11359687, 53077296)

In [29]:
final_df.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53077296 entries, 0 to 53077295
Data columns (total 84 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Flow ID            object 
 1   Src IP             object 
 2   Src Port           float64
 3   Dst IP             object 
 4   Dst Port           float64
 5   Protocol           float64
 6   Timestamp          object 
 7   Flow Duration      float64
 8   Tot Fwd Pkts       float64
 9   Tot Bwd Pkts       float64
 10  TotLen Fwd Pkts    float64
 11  TotLen Bwd Pkts    float64
 12  Fwd Pkt Len Max    float64
 13  Fwd Pkt Len Min    float64
 14  Fwd Pkt Len Mean   float64
 15  Fwd Pkt Len Std    float64
 16  Bwd Pkt Len Max    float64
 17  Bwd Pkt Len Min    float64
 18  Bwd Pkt Len Mean   float64
 19  Bwd Pkt Len Std    float64
 20  Flow Byts/s        float64
 21  Flow Pkts/s        float64
 22  Flow IAT Mean      float64
 23  Flow IAT Std       float64
 24  Flow IAT Max       float64
 25  Flow IAT Min    

In [30]:
# For each dataset
for file, df in dfs.items():
    unique_flows = df['Flow ID'].nunique()
    total_rows = df.shape[0]
    print(f"{file}: {unique_flows} unique Flow IDs, {total_rows} total rows")


Bot_IoT_DDoS.csv: 2102402 unique Flow IDs, 38049077 total rows
CombineFileAnCa171813.csv: 9257286 unique Flow IDs, 15028219 total rows
